# Role 3: IndicTrans2 Translation Test (Google Colab T4 GPU)
**Project:** Generative AI Based Multilingual Video Conferencing (`multilingual-meet`)

**Instructions:**
1. Go to **Runtime > Change runtime type**, select **T4 GPU**, and click **Save**.
2. Run each cell sequentially using the play button.

In [ ]:
# Cell 1: Install required libraries
!pip install -q transformers torch sentencepiece sacremoses sacrebleu jiwer accelerate

In [ ]:
# Cell 2: Hugging Face Login (if required for gated terms)
# Ensure you accepted terms at:
# https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
# https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 3: Load Models on GPU
import torch
import time
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")

EN_INDIC_ID = "ai4bharat/indictrans2-en-indic-dist-200M"
INDIC_EN_ID = "ai4bharat/indictrans2-indic-en-dist-200M"

print("Loading En -> Indic model...")
t0 = time.time()
en_indic_tokenizer = AutoTokenizer.from_pretrained(EN_INDIC_ID, trust_remote_code=True)
en_indic_model = AutoModelForSeq2SeqLM.from_pretrained(
    EN_INDIC_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
en_indic_model.eval()
print(f"En -> Indic loaded in {time.time() - t0:.2f}s")

print("Loading Indic -> En model...")
t1 = time.time()
indic_en_tokenizer = AutoTokenizer.from_pretrained(INDIC_EN_ID, trust_remote_code=True)
indic_en_model = AutoModelForSeq2SeqLM.from_pretrained(
    INDIC_EN_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
indic_en_model.eval()
print(f"Indic -> En loaded in {time.time() - t1:.2f}s")

In [ ]:
# Cell 4: Translation Function with English Pivoting
LANG_TAGS = {
    "en": "eng_Latn",
    "hi": "hin_Deva",
    "kn": "kan_Knda",
    "ta": "tam_Taml",
    "te": "tel_Telu",
    "ml": "mal_Mlym",
    "mr": "mar_Deva",
}

def direct_translate(text, src_code, tgt_code, is_en_to_indic=True):
    tokenizer = en_indic_tokenizer if is_en_to_indic else indic_en_tokenizer
    model = en_indic_model if is_en_to_indic else indic_en_model
    src_tag = LANG_TAGS[src_code]
    tgt_tag = LANG_TAGS[tgt_code]
    
    inputs = tokenizer([text], src_lang=src_tag, tgt_lang=tgt_tag, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, num_beams=2, max_length=256)
    return tokenizer.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)[0].strip()

def translate(text, src, tgt):
    if src == tgt or not text.strip():
        return text, 0.0
    start = time.time()
    if src == "en":
        res = direct_translate(text, src, tgt, is_en_to_indic=True)
    elif tgt == "en":
        res = direct_translate(text, src, tgt, is_en_to_indic=False)
    else:
        # Pivot via English
        intermediate = direct_translate(text, src, "en", is_en_to_indic=False)
        res = direct_translate(intermediate, "en", tgt, is_en_to_indic=True)
    latency_ms = (time.time() - start) * 1000.0
    return res, latency_ms

In [ ]:
# Cell 5: Benchmark & Test Sentences across en, hi, kn
test_cases = [
    ("en", "hi", "Hello everyone, welcome to the conference call."),
    ("en", "kn", "Hello everyone, welcome to the conference call."),
    ("hi", "en", "नमस्ते, आप सब कैसे हैं?"),
    ("kn", "en", "ಎಲ್ಲರಿಗೂ ನಮಸ್ಕಾರ, ಸಭೆಗೆ ಸ್ವಾಗತ."),
    ("hi", "kn", "आज का मौसम बहुत अच्छा है।") # Pivoting through English
]

print("=" * 75)
print(f"{'Pair':<10} | {'Input Text':<30} | {'Translation':<25} | {'Latency (ms)'}")
print("=" * 75)

for src, tgt, text in test_cases:
    output, lat = translate(text, src, tgt)
    print(f"{src}->{tgt:<7} | {text:<30} | {output:<25} | {lat:.1f} ms")

print("=" * 75)
print("All test cases passed successfully!")